# 02 — Annotation preparation, validation and BIO dataset (PackSure)

**Input:** reviewed annotation JSONs in the canonical char-span format:

```json
{
  "id": "ann-img1.jpg",
  "product_id": "maggi-noodles-70g",
  "text": "MRP Rs.120/- Net Qty 500G",
  "source_image": "img1.jpg",
  "reviewed": true,
  "entities": [
    {"start": 0, "end": 12, "label": "MRP"},
    {"start": 13, "end": 25, "label": "NET_QUANTITY"}
  ]
}
```

**What this notebook does:** validates your annotations (Phase 18 checks), reports label balance, converts to BIO, augments OCR noise, splits by product (leak-free), writes train/validation/test JSONL.

**What YOU do:** annotate — by editing JSON in Colab (mini-helper below) or in Label Studio. Mark `"reviewed": true` only for samples YOU verified.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys
from pathlib import Path
REPO = Path('/content/packsure')  # clone or upload ml/ here
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ShubhamKumar1729/packsure.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'ml' / 'scripts'))
ANN_DIR = Path('/content/drive/MyDrive/packsure/ml/dataset/annotations/approved')  # move reviewed files here
print('Annotation dir:', ANN_DIR)

In [ ]:
# Optional mini-annotator: compute char spans by selecting a substring.
# (For bulk annotation use Label Studio with the same char-span export; see
# ml/ANNOTATION_GUIDELINES.md.)
def span_for(text: str, substring: str, label: str, occurrence: int = 1) -> dict:
    starts = [i for i in range(len(text)) if text.startswith(substring, i)]
    assert occurrence <= len(starts), f'occurrence {occurrence} not found'
    start = starts[occurrence - 1]
    return {"start": start, "end": start + len(substring), "label": label}

# Example on your own text (the example text is a FORMAT demo, not real data):
demo = "MRP Rs.120/- Net Qty 500G"
print(span_for(demo, "Rs.120/-", "MRP"))
print(span_for(demo, "Net Qty 500G", "NET_QUANTITY"))

In [ ]:
# Validate everything (Phase 18): unknown labels, invalid/overlapping spans,
# empty texts, duplicates, label balance, never-annotated labels.
import json
from annotation_io import load_directory, dataset_report

samples = load_directory(ANN_DIR)
report = dataset_report(samples)
print(json.dumps({k: v for k, v in report.items() if k != 'issues'}, indent=2))
for issue in report['issues'][:30]:
    print('FIX:', issue)
assert report['invalid_samples'] == 0, 'Fix the issues above before training.'
assert report['total_samples'] > 0, 'No annotations found.'
print('\nAll annotations valid. Reviewed:', report['reviewed_samples'])

In [ ]:
# Train/validation/test split at PRODUCT level (no near-duplicate leakage),
# then OCR-noise augmentation of TRAIN ONLY (ground truth stays untouched).
from split import split_samples, check_no_leakage
from augmentation import augment_sample
import random

records = [
    {"id": s.id, "product_id": s.product_id, "text": s.text,
     "entities": [e for e in s.entities if isinstance(e, dict)], "source_image": s.source_image}
    for s in samples if s.valid
]
splits = split_samples(records, train_ratio=0.8, validation_ratio=0.1, seed=42)
leaks = check_no_leakage(splits)
assert not leaks, leaks

rng = random.Random(42)
augmented = []
for record in splits['train']:
    augmented.extend(augment_sample(record, rng, copies=2))
splits['train'] = splits['train'] + augmented
print({k: len(v) for k, v in splits.items()})
print(f'augmented train with {len(augmented)} OCR-noise copies (originals preserved)')

In [ ]:
# Convert to BIO token records and write JSONL datasets for training.
from bio import token_records_to_bio_jsonl, LABEL_LIST, LABEL_TO_ID
from annotation_io import save_jsonl, Sample, validate_sample
from pathlib import Path

OUT = Path('/content/drive/MyDrive/packsure/ml/dataset')
for name, split_samples in splits.items():
    rows = token_records_to_bio_jsonl(split_samples)
    target = OUT / name / f'{name}.jsonl'
    target.parent.mkdir(parents=True, exist_ok=True)
    with open(target, 'w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(name, '->', target, f'({len(rows)} rows)')

# Export the label mapping the trainer and the serving side both consume.
mapping = {"labels": LABEL_LIST, "label_to_id": LABEL_TO_ID,
           "id_to_label": {str(v): k for k, v in LABEL_TO_ID.items()}}
(OUT / 'label_mapping.json').write_text(json.dumps(mapping, indent=2))
print('label_mapping.json written with', len(LABEL_LIST), 'labels')

**Checkpoint:** `train/validation/test` JSONL + `label_mapping.json` exist on Drive.
Test stays untouched until notebook 04.